# Основы машинного обучения: классификация с помощью sklearn

**Тема**: бинарная классификация — предсказание выживаемости персонажей «Игры Престолов».

В этом конспекте разберём полный pipeline ML-задачи:
1. Загрузка и анализ данных (EDA)
2. Предобработка и feature engineering
3. Обучение и сравнение моделей
4. Оценка качества и подбор гиперпараметров

## 1. Задача классификации

Задача классификации — это задача машинного обучения с учителем (supervised learning), в которой модель предсказывает **дискретную метку класса** для каждого объекта.

В нашем случае — **бинарная классификация**: жив персонаж (`isAlive=1`) или мёртв (`isAlive=0`).

### Метрика Accuracy

$$\text{Accuracy} = \frac{\text{Число верных предсказаний}}{\text{Общее число предсказаний}}$$

Простая и интуитивная метрика, но может быть обманчивой при дисбалансе классов. Например, если 78% персонажей живы, модель-"заглушка", предсказывающая всем `isAlive=1`, получит Accuracy = 0.78.

### Pipeline машинного обучения

Типичный pipeline:
1. **Загрузка и анализ данных** (EDA)
2. **Предобработка** (очистка, заполнение пропусков, кодирование)
3. **Выбор и обучение моделей**
4. **Оценка качества**
5. **Применение к тестовым данным**

## 2. Данные

**Датасет**: персонажи вселенной «Игра Престолов» из вики A Wiki of Ice and Fire.

| Параметр | Значение |
|----------|----------|
| Train | 1557 записей, 25 признаков |
| Test | 389 записей, 24 признака (без target) |
| Target | `isAlive` (0 — мёртв, 1 — жив) |
| Баланс классов | ~78% живы, ~22% мертвы |

### Основные признаки
- `name`, `title`, `house`, `culture` — текстовые/категориальные
- `male`, `isMarried`, `isNoble` — бинарные
- `book1`–`book5` — появление в книгах (бинарные)
- `popularity` — числовой (0–1)
- `age`, `dateOfBirth` — числовые с большим количеством пропусков
- `isAliveMother/Father/Heir/Spouse` — бинарные с пропусками (>87%)

### Особенности
Очень много пропущенных значений (до 98.8% в некоторых столбцах), нельзя удалять строки с NaN.

## 3. Предобработка

### Работа с пропусками
- **Столбцы с >85% пропусков** (`mother`, `father`, `heir`, `spouse` и их `isAlive*` аналоги): заполняем NaN нулями после one-hot кодирования
- **`age`**: создаём два признака — `age_value` (возраст или 0) и `age_no_data` (флаг отсутствия данных). Это лучше, чем просто заполнение средним, т.к. сам факт отсутствия возраста информативен
- **`dateOfBirth`**: не включаем отдельно, т.к. `age_no_data` и `dateOfBirth_no_data` полностью совпадают

### Feature engineering

| Преобразование | Обоснование |
|----------------|-------------|
| `log10(popularity * 100 + 1)` | Сильно скошенное распределение → логарифм делает его ближе к нормальному |
| `boolDeadRelations` из `numDeadRelations` | У 95% персонажей значение = 0, бинарный признак информативнее |
| Группировка `culture` в 12 категорий | Слишком много уникальных значений (51), многие дублируются |
| Группировка `house` (топ-15 + Other) | 315 уникальных домов → слишком большая размерность при one-hot |

### Кодирование категориальных переменных
- **One-hot encoding** (`pd.get_dummies`) для `culture_grouped` и `house_grouped`
- `drop_first=True` для избежания мультиколлинеарности в линейных моделях
- Выравнивание столбцов train и test (добавление отсутствующих, удаление лишних)

### Удалённые столбцы
- `name` — идентификатор, не признак
- `title`, `spouse`, `father`, `mother`, `heir` — высокая кардинальность, мало данных
- `DateOfDeath` — **data leakage** (почти дублирует target)

### Примеры кода: pandas

In [ ]:
import pandas as pd
import numpy as np

# Загрузка и первичный анализ
# df = pd.read_csv('data/game_of_thrones_train.csv', index_col='S.No')
# df.info()                           # Типы данных и пропуски
# df.describe(include='object').T     # Статистика категориальных
# df.isnull().sum()                   # Подсчёт пропусков
# df['col'].value_counts(dropna=False) # Частоты с учётом NaN

# Нормализация строк и маппинг
# df['col'].str.lower().str.strip()
# df['col'].map(dict)

# One-hot кодирование
# pd.get_dummies(df, columns=['col1', 'col2'], drop_first=True)

print('Примеры pandas-операций — см. комментарии выше')

## 4. Модели

### Logistic Regression
- **Принцип**: линейная модель, моделирует вероятность класса через сигмоиду
- **Плюсы**: быстрая, интерпретируемая, хороша как baseline
- **Минусы**: предполагает линейную разделимость
- **Требует**: масштабирование признаков (StandardScaler)
- **Результат**: Accuracy = 0.7917

### Random Forest
- **Принцип**: ансамбль решающих деревьев, каждое обучается на случайной подвыборке
- **Плюсы**: устойчив к переобучению, не требует масштабирования, показывает feature importance
- **Минусы**: много гиперпараметров, может быть медленным
- **Гиперпараметры**: `n_estimators=200`, `max_depth=10`
- **Результат**: Accuracy = 0.8109

### Gradient Boosting
- **Принцип**: последовательное построение деревьев, каждое исправляет ошибки предыдущих
- **Плюсы**: часто даёт лучшее качество, гибкий
- **Минусы**: медленнее RF, чувствителен к гиперпараметрам
- **Лучшие параметры** (GridSearchCV): `learning_rate=0.05`, `max_depth=3`, `n_estimators=200`
- **Результат**: Accuracy = 0.8077 (val), CV = 0.8189 ± 0.0245

### KNN (K-Nearest Neighbors)
- **Принцип**: классифицирует по k ближайшим соседям в пространстве признаков
- **Плюсы**: простой, нет фазы обучения
- **Минусы**: медленный на больших данных, чувствителен к масштабу и выбору k
- **Требует**: масштабирование признаков
- **Результат**: Accuracy = 0.7853

### SVM (Support Vector Machine)
- **Принцип**: ищет гиперплоскость, максимально разделяющую классы
- **Плюсы**: эффективен в высокоразмерных пространствах
- **Минусы**: медленный на больших данных, чувствителен к масштабу
- **Требует**: масштабирование признаков
- **Результат**: Accuracy = 0.7724

### Примеры кода: sklearn

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Разделение данных
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# Масштабирование
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)  # fit + transform на train
# X_val_scaled = scaler.transform(X_val)           # только transform на val/test

# Обучение и предсказание
# model.fit(X_train, y_train)
# y_pred = model.predict(X_val)

# Оценка
# accuracy_score(y_val, y_pred)
# cross_val_score(model, X, y, cv=5, scoring='accuracy')

# Подбор гиперпараметров
# grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# grid.fit(X_train, y_train)
# grid.best_params_, grid.best_score_

print('Примеры sklearn-операций — см. комментарии выше')

## 5. Результаты

| Модель | Accuracy (val) |
|--------|---------------|
| Random Forest | 0.8109 |
| Gradient Boosting | 0.8077 |
| Logistic Regression | 0.7917 |
| KNN | 0.7853 |
| SVM | 0.7724 |

**Лучшая модель**: Gradient Boosting (после GridSearchCV). Хотя Random Forest показал чуть лучший результат на валидации, GB стабильнее на cross-validation (CV Accuracy = 0.8189).

**Финальный результат**: Accuracy ≈ 0.82, что соответствует **5 баллам из 5** (порог ≥ 0.65).

**Топ-3 важных признака** (Gradient Boosting feature importance):
1. `popularity` — популярность персонажа
2. `boolDeadRelations` — наличие мёртвых родственников
3. `book4` — появление в 4-й книге

## 6. Уроки и выводы

1. **Data leakage** — признак `DateOfDeath` почти полностью дублирует target. Всегда проверяйте, нет ли признаков, которые "подсматривают" ответ.

2. **Пропуски ≠ мусор** — сам факт отсутствия данных может быть информативным (признак `age_no_data` оказался полезным).

3. **Группировка категорий** — при большом количестве уникальных значений группировка снижает размерность и уменьшает шум.

4. **Масштабирование** — обязательно для моделей, основанных на расстоянии (KNN, SVM, LogReg). Деревья и ансамбли деревьев масштабирования не требуют.

5. **GridSearchCV** — перебор гиперпараметров с cross-validation даёт более надёжную оценку, чем подбор на одном validation split.

6. **Воспроизводимость** — `random_state=42` во всех случайных операциях.

7. **Одинаковый preprocessing для train и test** — нельзя fit scaler/encoder на test данных.